In [175]:
import sys
import os
from dotenv import load_dotenv

# 1. Dapatkan path absolut dari direktori saat ini (ml_models)
current_dir = os.path.abspath('')

# 2. Dapatkan path dari parent directory (root folder)
parent_dir = os.path.dirname(current_dir)

# 3. Tambahkan parent directory ke sys.path agar Python bisa menemukan config.py
if parent_dir not in sys.path:
    sys.path.append(parent_dir)

# 4. Load file .env secara eksplisit dari root folder agar DB_USER dkk terbaca
load_dotenv(os.path.join(parent_dir, '.env'))

True

In [176]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import RobustScaler, StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, calinski_harabasz_score
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import optuna

import lightgbm as lgb
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import TimeSeriesSplit

from config import engine

In [177]:
def fetch_data():
    print("Fetching data from database...")
    query = """
        SELECT 
            d.date, 
            d.days_name, 
            d.month, 
            d.year, 
            d.is_weekend, 
            d.is_payday, 
            d.is_ramadhan, 
            pr.product_model, 
            pr.product_color AS color, 
            pr.product_size AS size, 
            pl.platform_name AS platform, 
            pm.payment_method_name AS payment_method, 
            l.province, 
            l.city, 
            f.quantity, 
            f.price, 
            f.discount, 
            f.total_amount
        FROM order_fact f
        LEFT JOIN date_dimension d ON f.date_id = d.date_id
        LEFT JOIN product_dimension pr ON f.product_id = pr.product_id
        LEFT JOIN platform_dimension pl ON f.platform_id = pl.platform_id
        LEFT JOIN payment_method_dimension pm ON f.payment_method_id = pm.payment_method_id
        LEFT JOIN location_dimension l ON f.location_id = l.location_id
        ORDER BY d.date DESC;
    """ 
    df = pd.read_sql(query, engine)
    print(f"Data loaded successfully! Shape: {df.shape}")
    return df

df_raw = fetch_data()

Fetching data from database...
2026-05-09 14:15:43,221 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-05-09 14:15:43,300 INFO sqlalchemy.engine.Engine DESCRIBE `janus_dw`.`
        SELECT 
            d.date, 
            d.days_name, 
            d.month, 
            d.year, 
            d.is_weekend, 
            d.is_payday, 
            d.is_ramadhan, 
            pr.product_model, 
            pr.product_color AS color, 
            pr.product_size AS size, 
            pl.platform_name AS platform, 
            pm.payment_method_name AS payment_method, 
            l.province, 
            l.city, 
            f.quantity, 
            f.price, 
            f.discount, 
            f.total_amount
        FROM order_fact f
        LEFT JOIN date_dimension d ON f.date_id = d.date_id
        LEFT JOIN product_dimension pr ON f.product_id = pr.product_id
        LEFT JOIN platform_dimension pl ON f.platform_id = pl.platform_id
        LEFT JOIN payment_method_dimension pm ON f.p

In [178]:
# Cell 2: Preprocessing & SKU Definition
df = df_raw.copy()

# Ensure datetime format
df['date'] = pd.to_datetime(df['date'])

# SKU level: Model + Warna (mengurangi intermittent demand)
df['sku'] = df['product_model'].astype(str) + "_" + df['color'].astype(str)

# Sort by date for safety
df = df.sort_values(by='date').reset_index(drop=True)

print(f"Total Unique SKUs: {df['sku'].nunique()}")

# 1. Aggregate per-SKU
sku_features = df.groupby('sku').agg(
    freq=('quantity', 'count'),
    total_qty=('quantity', 'sum'),
    median_qty=('quantity', 'median'),
    mean_price=('price', 'mean'),
    total_revenue=('total_amount', 'sum'),
    std_qty=('quantity', 'std')
).fillna(0) # Fill NaN for std_qty if freq is 1

# Coefficient of Variation (CV)
sku_features['cv_qty'] = np.where(sku_features['total_qty'] > 0, 
                                  sku_features['std_qty'] / (sku_features['total_qty']/sku_features['freq']), 0)

# 2. Log Transform for skewed variables
skewed_cols = ['total_qty', 'total_revenue', 'mean_price']
for col in skewed_cols:
    sku_features[f'log_{col}'] = np.log1p(sku_features[col])

# Drop original skewed columns
features_to_scale = [c for c in sku_features.columns if c not in skewed_cols]
X = sku_features[features_to_scale]

# 3. Scaling (RobustScaler handles outliers well)
scaler = RobustScaler()
X_scaled = scaler.fit_transform(X)

# Optional: PCA to reduce noise before K-Means
pca = PCA(n_components=0.95) # Keep 95% variance
X_pca = pca.fit_transform(X_scaled)

# 4. Find Best K using Silhouette & CH Score
best_k = 3
best_score = -1
metrics = []

for k in range(2, 8):
    kmeans = KMeans(n_clusters=k, n_init=10, random_state=42)
    labels = kmeans.fit_predict(X_pca)
    sil_score = silhouette_score(X_pca, labels)
    ch_score = calinski_harabasz_score(X_pca, labels)
    metrics.append({'k': k, 'silhouette': sil_score, 'ch_score': ch_score})
    
    if sil_score > best_score:
        best_score = sil_score
        best_k = k

print(pd.DataFrame(metrics))
print(f"\nSelected Optimal k: {best_k}")

# Final Clustering
final_kmeans = KMeans(n_clusters=best_k, n_init=10, random_state=42)
sku_features['cluster'] = final_kmeans.fit_predict(X_pca)

# Map clusters back to main dataframe
df = df.merge(sku_features[['cluster']], on='sku', how='left')

Total Unique SKUs: 267
   k  silhouette    ch_score
0  2    0.570505  170.454894
1  3    0.468529  197.102015
2  4    0.473554  224.788204
3  5    0.459732  218.872732
4  6    0.410659  243.836010
5  7    0.447109  289.684598

Selected Optimal k: 2


In [179]:
# 1. Aggregate per-SKU
sku_features = df.groupby('sku').agg(
    freq=('quantity', 'count'),
    total_qty=('quantity', 'sum'),
    median_qty=('quantity', 'median'),
    mean_price=('price', 'mean'),
    total_revenue=('total_amount', 'sum'),
    std_qty=('quantity', 'std')
).fillna(0) # Fill NaN for std_qty if freq is 1

# Coefficient of Variation (CV)
sku_features['cv_qty'] = np.where(sku_features['total_qty'] > 0, 
                                  sku_features['std_qty'] / (sku_features['total_qty']/sku_features['freq']), 0)

# 2. Log Transform for skewed variables (Winsorize via RobustScaler later)
skewed_cols = ['total_qty', 'total_revenue', 'mean_price']
for col in skewed_cols:
    sku_features[f'log_{col}'] = np.log1p(sku_features[col])

# Drop original skewed columns
features_to_scale = [c for c in sku_features.columns if c not in skewed_cols]
X = sku_features[features_to_scale]

# 3. Scaling (RobustScaler handles outliers well)
scaler = RobustScaler()
X_scaled = scaler.fit_transform(X)

# Optional: PCA to reduce noise before K-Means
pca = PCA(n_components=0.95) # Keep 95% variance
X_pca = pca.fit_transform(X_scaled)

# 4. Find Best K using Silhouette & CH Score
best_k = 3
best_score = -1
metrics = []

for k in range(2, 8):
    kmeans = KMeans(n_clusters=k, n_init=10, random_state=42)
    labels = kmeans.fit_predict(X_pca)
    sil_score = silhouette_score(X_pca, labels)
    ch_score = calinski_harabasz_score(X_pca, labels)
    metrics.append({'k': k, 'silhouette': sil_score, 'ch_score': ch_score})
    
    if sil_score > best_score:
        best_score = sil_score
        best_k = k

print(pd.DataFrame(metrics))
print(f"\nSelected Optimal k: {best_k}")

# Final Clustering
final_kmeans = KMeans(n_clusters=best_k, n_init=10, random_state=42)
sku_features['cluster'] = final_kmeans.fit_predict(X_pca)

# Map clusters back to main dataframe
df = df.merge(sku_features[['cluster']], on='sku', how='left')

   k  silhouette    ch_score
0  2    0.570505  170.454894
1  3    0.468529  197.102015
2  4    0.473554  224.788204
3  5    0.459732  218.872732
4  6    0.410659  243.836010
5  7    0.447109  289.684598

Selected Optimal k: 2


In [180]:
# Cell 4: Filter Fast Cluster, Weekly Aggregation & Feature Engineering

print("--- FILTER CLUSTER FAST ---")
# Cari tahu otomatis mana cluster yang 'Fast' berdasarkan rata-rata qty
cluster_means = sku_features.groupby('cluster')['total_qty'].mean()
fast_cluster = cluster_means.idxmax()
print(f"Menggunakan Cluster {fast_cluster} (Fast Moving) dan membuang sisanya.")

# Ambil list SKU yang masuk kategori fast
fast_skus = sku_features[sku_features['cluster'] == fast_cluster].index

# FILTER DATA RAW SEBELUM DIAGREGASI
df_filtered = df[df['sku'].isin(fast_skus)].copy()
df_filtered = df_filtered[df_filtered['sku'] != 'UNKNOWN_UNKNOWN']

# 1. Agregasi dasar ke weekly
weekly_df_raw = df_filtered.groupby(['sku', pd.Grouper(key='date', freq='W-MON')]).agg(
    weekly_qty=('quantity', 'sum'),
    avg_price=('price', 'mean'),
    avg_discount=('discount', 'mean'),
    is_payday=('is_payday', 'max'),
    is_ramadhan=('is_ramadhan', 'max')
).reset_index()

# 2. BIKIN DATA BERUNTUN (Zero-Filling)
min_date = weekly_df_raw['date'].min()
max_date = weekly_df_raw['date'].max()
all_weeks = pd.date_range(start=min_date, end=max_date, freq='W-MON')

unique_skus = weekly_df_raw['sku'].unique()
multi_idx = pd.MultiIndex.from_product([unique_skus, all_weeks], names=['sku', 'date'])
full_grid = pd.DataFrame(index=multi_idx).reset_index()

# Gabungkan grid kosong dengan data agregasi raw
weekly_df = pd.merge(full_grid, weekly_df_raw, on=['sku', 'date'], how='left')

# 3. Handling Nilai Kosong (Imputation)
weekly_df['weekly_qty'] = weekly_df['weekly_qty'].fillna(0)
weekly_df['avg_discount'] = weekly_df['avg_discount'].fillna(0)
weekly_df['is_payday'] = weekly_df['is_payday'].fillna(0)
weekly_df['is_ramadhan'] = weekly_df['is_ramadhan'].fillna(0)

# Imputasi harga: ffill lalu bfill
weekly_df['avg_price'] = weekly_df.groupby('sku')['avg_price'].transform(lambda x: x.ffill().bfill())

# Kembalikan mapping cluster (sekarang semuanya seragam fast_cluster)
weekly_df['cluster'] = fast_cluster

# 4. Generate Time-Series Features
def create_ts_features(group):
    for lag in [1, 2, 3, 4]:
        group[f'qty_lag_{lag}'] = group['weekly_qty'].shift(lag)
    
    group['rolling_mean_4'] = group['weekly_qty'].shift(1).rolling(window=4, min_periods=1).mean()
    group['rolling_mean_8'] = group['weekly_qty'].shift(1).rolling(window=8, min_periods=1).mean()
    group['price_diff'] = group['avg_price'].diff()
    return group

weekly_df = weekly_df.sort_values(['sku', 'date'])
weekly_df = weekly_df.groupby('sku', group_keys=False).apply(create_ts_features)

# Calendar features
weekly_df['month'] = weekly_df['date'].dt.month
weekly_df['weekofyear'] = weekly_df['date'].dt.isocalendar().week.astype(int)

# Drop NaN akibat lagging
weekly_df.dropna(inplace=True)
weekly_df = weekly_df.reset_index(drop=True)

# 5. DEKLARASI TARGET & FEATURES (Agar bisa dipakai Optuna di sel berikutnya)
TARGET = 'weekly_qty'
ignore_cols = ['sku', 'date', TARGET, 'cluster', 'month_year', 'predicted_weekly_qty']
FEATURES = [c for c in weekly_df.columns if c not in ignore_cols]

print(f"Data siap! Total baris: {len(weekly_df)} dengan {len(unique_skus)} SKU Fast Moving.")

--- FILTER CLUSTER FAST ---
Menggunakan Cluster 1 (Fast Moving) dan membuang sisanya.
Data siap! Total baris: 3224 dengan 31 SKU Fast Moving.


In [181]:
# Cell 6: Hyperparameter Tuning with Optuna

def objective(trial):
    # Holdout validation (last 20% of time) for fast tuning
    split_idx = int(len(weekly_df) * 0.8)
    train_df = weekly_df.iloc[:split_idx]
    val_df = weekly_df.iloc[split_idx:]
    
    # Tidak lagi menggunakan categorical_feature=['cluster']
    train_set = lgb.Dataset(train_df[FEATURES], train_df[TARGET])
    val_set = lgb.Dataset(val_df[FEATURES], val_df[TARGET], reference=train_set)
    
    params = {
        'objective': 'regression',
        'metric': 'mae',
        'boosting_type': 'gbdt',
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 20, 100),
        'max_depth': trial.suggest_int('max_depth', 5, 15),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.6, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.6, 1.0),
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 7),
        'lambda_l1': trial.suggest_float('lambda_l1', 1e-8, 10.0, log=True),
        'lambda_l2': trial.suggest_float('lambda_l2', 1e-8, 10.0, log=True),
        'verbose': -1
    }
    
    model = lgb.train(
        params, train_set, valid_sets=[val_set], num_boost_round=500,
        callbacks=[lgb.early_stopping(stopping_rounds=30, verbose=False)]
    )
    
    preds = model.predict(val_df[FEATURES])
    preds = np.clip(preds, 0, None)
    return mean_absolute_error(val_df[TARGET], preds)

# Uncomment to run optimization
# study = optuna.create_study(direction='minimize')
# study.optimize(objective, n_trials=20)
# print('Best params:', study.best_params)

In [182]:
# Cell 7: Validation (Best Params), Final Training & Sanity Check

import lightgbm as lgb
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import TimeSeriesSplit

# 1. PARAMETER TERBAIK (Hasil Optuna)
best_params = {
    'objective': 'regression',
    'metric': 'mae',
    'boosting_type': 'gbdt',
    'learning_rate': 0.09891950177299384,
    'num_leaves': 85,
    'max_depth': 9,
    'feature_fraction': 0.8065919912947699,
    'bagging_fraction': 0.747756016272759,
    'bagging_freq': 6,
    'lambda_l1': 0.42153532644108826,
    'lambda_l2': 0.00018850776777126722,
    'random_state': 42,
    'verbose': -1
}

# 2. VALIDASI (TIME-SERIES CV)
print("--- 1. TIME-SERIES CROSS VALIDATION ---")
tscv = TimeSeriesSplit(n_splits=5)
fold_metrics = []

for fold, (train_idx, val_idx) in enumerate(tscv.split(weekly_df)):
    train_data = weekly_df.iloc[train_idx]
    val_data = weekly_df.iloc[val_idx]
    
    X_train, y_train = train_data[FEATURES], train_data[TARGET]
    X_val, y_val = val_data[FEATURES], val_data[TARGET]
    
    train_set = lgb.Dataset(X_train, y_train)
    val_set = lgb.Dataset(X_val, y_val, reference=train_set)
    
    model = lgb.train(
        best_params, train_set, valid_sets=[train_set, val_set],
        num_boost_round=1000, callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
    )
    
    preds = model.predict(X_val)
    preds = np.clip(preds, 0, None)
    
    mae = mean_absolute_error(y_val, preds)
    wape = np.sum(np.abs(y_val - preds)) / np.sum(y_val) if np.sum(y_val) > 0 else 0
    fold_metrics.append({'fold': fold, 'mae': mae, 'wape': wape})
    print(f"Fold {fold} | MAE: {mae:.2f} | WAPE: {wape:.2%}")

print("\nRata-rata Metrik Validasi:")
print(pd.DataFrame(fold_metrics).mean())

# 3. MELATIH MODEL FINAL
print("\n--- 2. MELATIH MODEL FINAL ---")
final_train_set = lgb.Dataset(weekly_df[FEATURES], label=weekly_df[TARGET])
final_model = lgb.train(best_params, final_train_set, num_boost_round=400)
print("=> Model Final berhasil dilatih!")

# 4. SANITY CHECK (Historis 4 Minggu Terakhir)
print("\n--- 3. SANITY CHECK: Forecast 1 Bulan Terakhir (Aggregated) ---")
top_3_skus = weekly_df.groupby('sku')[TARGET].sum().nlargest(3).index.tolist()
last_4_weeks = sorted(weekly_df['date'].unique())[-4:]

sample_data = weekly_df[(weekly_df['sku'].isin(top_3_skus)) & (weekly_df['date'].isin(last_4_weeks))].copy()
sample_preds = final_model.predict(sample_data[FEATURES])
sample_data['predicted_weekly_qty'] = np.clip(sample_preds, 0, None)

sample_data['month_year'] = sample_data['date'].dt.to_period('M').astype(str)
monthly_forecast = sample_data.groupby(['sku', 'month_year']).agg(
    total_actual_qty=(TARGET, 'sum'),
    total_predicted_qty=('predicted_weekly_qty', 'sum')
).reset_index()

monthly_forecast['total_predicted_qty'] = monthly_forecast['total_predicted_qty'].round(0).astype(int)
monthly_forecast['total_actual_qty'] = monthly_forecast['total_actual_qty'].astype(int)

print(monthly_forecast.to_string(index=False))

--- 1. TIME-SERIES CROSS VALIDATION ---
Fold 0 | MAE: 1.52 | WAPE: 57.90%
Fold 1 | MAE: 1.30 | WAPE: 50.50%
Fold 2 | MAE: 1.82 | WAPE: 49.95%
Fold 3 | MAE: 1.42 | WAPE: 45.45%
Fold 4 | MAE: 0.97 | WAPE: 50.46%

Rata-rata Metrik Validasi:
fold    2.000000
mae     1.406443
wape    0.508541
dtype: float64

--- 2. MELATIH MODEL FINAL ---
=> Model Final berhasil dilatih!

--- 3. SANITY CHECK: Forecast 1 Bulan Terakhir (Aggregated) ---
             sku month_year  total_actual_qty  total_predicted_qty
AERA_COKELAT TUA    2026-04                21                   19
AERA_COKELAT TUA    2026-05                 3                    3
   NINA_BURGUNDY    2026-04                 2                    2
   NINA_BURGUNDY    2026-05                 0                    0
 SEQUIN_BURGUNDY    2026-04                15                   14
 SEQUIN_BURGUNDY    2026-05                 2                    2


In [183]:
train_data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2687 entries, 0 to 2686
Data columns (total 17 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   sku             2687 non-null   object        
 1   date            2687 non-null   datetime64[ns]
 2   weekly_qty      2687 non-null   float64       
 3   avg_price       2687 non-null   float64       
 4   avg_discount    2687 non-null   float64       
 5   is_payday       2687 non-null   float64       
 6   is_ramadhan     2687 non-null   float64       
 7   cluster         2687 non-null   int32         
 8   qty_lag_1       2687 non-null   float64       
 9   qty_lag_2       2687 non-null   float64       
 10  qty_lag_3       2687 non-null   float64       
 11  qty_lag_4       2687 non-null   float64       
 12  rolling_mean_4  2687 non-null   float64       
 13  rolling_mean_8  2687 non-null   float64       
 14  price_diff      2687 non-null   float64       
 15  month    

In [184]:
train_data.head()

,sku,date,weekly_qty,avg_price,avg_discount,is_payday,is_ramadhan,cluster,qty_lag_1,qty_lag_2,qty_lag_3,qty_lag_4,rolling_mean_4,rolling_mean_8,price_diff,month,weekofyear
0,AERA LONG DRESS_TERRACOTTA,2024-05-13,0.0,97000.0,0.0,0.0,0.0,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5,20
1,AERA LONG DRESS_TERRACOTTA,2024-05-20,0.0,97000.0,0.0,0.0,0.0,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5,21
2,AERA LONG DRESS_TERRACOTTA,2024-05-27,0.0,97000.0,0.0,0.0,0.0,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5,22
3,AERA LONG DRESS_TERRACOTTA,2024-06-03,0.0,97000.0,0.0,0.0,0.0,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6,23
4,AERA LONG DRESS_TERRACOTTA,2024-06-10,0.0,97000.0,0.0,0.0,0.0,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6,24


In [185]:
# Cell 8: Future Forecasting (Retail Calendar / Mayoritas)

print("\n--- FORECAST REKURSIF BULAN MEI 2026 ---")

# Ambil data aktual minggu TERAKHIR di dataset sebagai batu pijakan awal
last_known_data = weekly_df.sort_values('date').groupby('sku').tail(1).copy()
future_predictions = []

# Gunakan 5 STEPS (35 hari) untuk memastikan seluruh hari di bulan target ter-cover
STEPS = 5 

for step in range(1, STEPS + 1):
    future_step = last_known_data.copy()
    
    # Geser kalender 7 hari ke depan
    future_step['date'] = future_step['date'] + pd.Timedelta(days=7)
    future_step['month'] = future_step['date'].dt.month
    future_step['weekofyear'] = future_step['date'].dt.isocalendar().week.astype(int)
    
    # Perbarui Lags secara kronologis
    future_step['qty_lag_4'] = future_step['qty_lag_3']
    future_step['qty_lag_3'] = future_step['qty_lag_2']
    future_step['qty_lag_2'] = future_step['qty_lag_1']
    # lag_1 selalu diisi dengan "weekly_qty" dari baris sebelumnya (aktual atau prediksi)
    future_step['qty_lag_1'] = future_step['weekly_qty'] 
    
    # Update rolling mean secara dinamis
    future_step['rolling_mean_4'] = (future_step['qty_lag_1'] + future_step['qty_lag_2'] + 
                                     future_step['qty_lag_3'] + future_step['qty_lag_4']) / 4
    
    # Eksekusi Prediksi untuk minggu ini
    preds = final_model.predict(future_step[FEATURES])
    future_step['weekly_qty'] = np.clip(preds, 0, None)
    
    future_predictions.append(future_step)
    last_known_data = future_step.copy()

# Gabungkan hasil 5 minggu ke depan
future_df = pd.concat(future_predictions)


# === AGREGASI BULANAN DENGAN LOGIKA MAYORITAS (RETAIL CALENDAR) ===

# 1. Cari titik tengah minggu tersebut (Geser mundur 3 hari dari hari Senin ke hari Jumat)
future_df['midpoint_date'] = future_df['date'] - pd.Timedelta(days=3)

# 2. Tentukan bulan berdasarkan hari Jumat tersebut
future_df['month_year'] = future_df['midpoint_date'].dt.to_period('M').astype(str)

# 3. Lakukan pengelompokan (Agregasi)
monthly_future = future_df.groupby(['sku', 'month_year']).agg(
    total_forecast_qty=('weekly_qty', 'sum')
).reset_index()

# Bulatkan hasil menjadi satuan unit
monthly_future['total_forecast_qty'] = monthly_future['total_forecast_qty'].round(0).astype(int)

# 4. Filter HANYA UNTUK BULAN MEI 2026 (Sembunyikan limpahan ke bulan Juni)
monthly_future_mei = monthly_future[monthly_future['month_year'] == '2026-05']

print("=> Hasil Forecast Final Bulan Mei 2026 (Menampilkan 15 SKU Teratas):")
print(monthly_future_mei.head(15).to_string(index=False))


--- FORECAST REKURSIF BULAN MEI 2026 ---


=> Hasil Forecast Final Bulan Mei 2026 (Menampilkan 15 SKU Teratas):
                       sku month_year  total_forecast_qty
AERA LONG DRESS_TERRACOTTA    2026-05                   1
            AERA_BIRU MUDA    2026-05                   1
             AERA_BURGUNDY    2026-05                   3
          AERA_COKELAT TUA    2026-05                  18
                AERA_HITAM    2026-05                  16
               AERA_MAROON    2026-05                   1
                 AERA_NAVY    2026-05                   2
            AERA_ROSE GOLD    2026-05                   8
                 AERA_SAGE    2026-05                   2
               AERA_SILVER    2026-05                   1
            AURORA_EMERALD    2026-05                   4
               DK_BURGUNDY    2026-05                   5
                  DK_HITAM    2026-05                   6
      M ADHEYYA_TERRACOTTA    2026-05                   1
        MELLA_COKELAT SUSU    2026-05                   1


In [186]:
import warnings
warnings.filterwarnings('ignore') # Biar gak muncul warning SettingWithCopy dari Pandas

# === TAMBAHAN: EXPORT SEMUA SKU & REKAP PER WARNA ===

# 1. Export ke Excel agar lu bisa lihat SEMUA SKU (tidak cuma 15)
# File ini akan otomatis terbuat di folder lu
# monthly_future_mei.to_excel("forecast_sku_mei_2026.xlsx", index=False)
print(f"\n=> Seluruh forecast untuk {len(monthly_future_mei)} SKU telah disimpan ke 'forecast_sku_mei_2026.xlsx'")

# 2. Ekstrak Warna dari kolom SKU (Format: "MODEL_WARNA")
# Kita ambil kata setelah underscore '_' menggunakan split
monthly_future_mei['warna'] = monthly_future_mei['sku'].apply(lambda x: x.split('_')[-1] if '_' in x else 'Unknown')

# 3. Agregasi (Total) Kebutuhan Baju Berdasarkan Warna
rekap_warna = monthly_future_mei.groupby('warna')['total_forecast_qty'].sum().reset_index()

# Urutkan dari warna dengan permintaan terbanyak ke terdikit
rekap_warna = rekap_warna.sort_values(by='total_forecast_qty', ascending=False).reset_index(drop=True)

print("\n=> REKAP KEBUTUHAN PER WARNA (Bulan Mei 2026):")
print(rekap_warna.to_string(index=False))

# (Opsional) Export rekap warna ke Excel juga
# rekap_warna.to_excel("rekap_warna_mei_2026.xlsx", index=False)


=> Seluruh forecast untuk 31 SKU telah disimpan ke 'forecast_sku_mei_2026.xlsx'

=> REKAP KEBUTUHAN PER WARNA (Bulan Mei 2026):
       warna  total_forecast_qty
 COKELAT TUA                  28
       HITAM                  24
    BURGUNDY                  19
     EMERALD                  15
    LAVENDER                  10
   ROSE GOLD                   8
COKELAT SUSU                   5
        SAGE                   4
  TERRACOTTA                   3
        NAVY                   3
EMERALD BLUE                   1
   BIRU MUDA                   1
      MAROON                   1
      SILVER                   1
